# 03 — Querying and exporting

You have a populated warehouse and you understand its schema. Now it's
time to actually *use* the data:

- **Three worked queries** in pandas that answer questions about the
  World Climbing competition history.
- The **seven pre-joined export views** that ship with the package
  (plus an opt-in `ascents` view) — what's in each one, when to
  regenerate them, how to read them back.

By the end you'll have a `data/exports/` directory full of CSVs ready
for downstream consumers.


## 1. Working directory + setup

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

from wcl_data.config import load_settings
from wcl_data.db.schema import open_db

settings = load_settings(require_credentials=False)
conn = open_db(settings.db_path)
print("DB:", settings.db_path)


## 2. pandas vs raw `sqlite3.Cursor`

The package's CLI and internals use plain `sqlite3` — no pandas
dependency, deliberately, so the core package stays lean.

For *analysis* though, pandas is the right tool. `pd.read_sql_query(sql, conn)`
gives you a `DataFrame` directly, and the SQLite driver handles the
type coercion (TEXT → str, INTEGER → int64, REAL → float64).

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 200)


## 3. Query 1: events per season

A simple aggregate over `events` with a join to `seasons` for the
year label. We use `LEFT JOIN` so events whose `season_id` is somehow
NULL still show up — a defensive choice.

In [ ]:
q = """
SELECT s.year       AS season_year,
       COUNT(e.id)  AS n_events
  FROM events e
  LEFT JOIN seasons s ON s.id = e.season_id
 GROUP BY s.year
 ORDER BY s.year
"""
events_per_year = pd.read_sql_query(q, conn)
events_per_year


### Quick stats

A `DataFrame` makes one-liner summaries easy.

In [ ]:
print(f"Years covered:    {events_per_year['season_year'].min():.0f}–{events_per_year['season_year'].max():.0f}")
print(f"Total events:     {events_per_year['n_events'].sum()}")
print(f"Median per year:  {events_per_year['n_events'].median():.0f}")
print(f"Peak year:        {events_per_year.loc[events_per_year['n_events'].idxmax(), 'season_year']:.0f} "
      f"({events_per_year['n_events'].max()} events)")


## 4. Query 2: most-active athletes

Counting how many competitions each athlete has appeared in tells you
who the long-career performers are. This joins
`results → competitions → athletes` (the typical traversal direction).

In [ ]:
q = """
SELECT a.firstname || ' ' || a.lastname  AS athlete,
       a.country,
       COUNT(DISTINCT r.competition_id)  AS n_competitions
  FROM results r
  JOIN athletes a ON a.id = r.athlete_id
 GROUP BY a.id
 ORDER BY n_competitions DESC
 LIMIT 20
"""
pd.read_sql_query(q, conn)


## 5. Query 3: country coverage of events over time

How widely distributed are events geographically? This counts the
number of distinct countries hosting at least one event per season.

In [ ]:
q = """
SELECT s.year                          AS season_year,
       COUNT(DISTINCT e.country)       AS n_countries,
       COUNT(e.id)                     AS n_events
  FROM events e
  LEFT JOIN seasons s ON s.id = e.season_id
 WHERE e.country IS NOT NULL
 GROUP BY s.year
 ORDER BY s.year
"""
coverage = pd.read_sql_query(q, conn)
coverage.tail(15)


These three queries are deliberately small — the warehouse is yours
to slice however you like. The schema is documented in chapter 01 if
you need a refresher on what joins to what.

## 6. The pre-joined export views

Writing the joins above by hand is fine for ad-hoc exploration, but
when you want to hand the data to a downstream consumer (a notebook
in a different repo, a BI tool, an Excel file) it's nicer to export
**flat, denormalized CSVs** that don't need foreign keys to follow.

The exporter at `src/wcl_data/exporter.py` defines eight such views.
The first seven are written by `export_all`; `ascents` is opt-in
because it's ~900k rows × 31 columns (~200 MB CSV).

| View | Default? | One row per | What's in it |
|------|:---:|-------------|--------------|
| `seasons` | ✓ | season | `season_ifsc_id`, `year`, `last_fetched_at` |
| `leagues` | ✓ | league | `league_id`, `name` |
| `events` | ✓ | event | event details + season year + league name + city + country + dates |
| `competitions` | ✓ | competition | event + season year + discipline + category + gender |
| `athletes` | ✓ | athlete | full profile, gender resolved to `"male"`/`"female"` |
| `results` | ✓ | (athlete × competition) | the *big one* — everything pre-joined, ready for analysis |
| `round_results` | ✓ | (athlete × round) | per-round rank/score/starting_group with full event context (ADR 0007) |
| `ascents` | opt-in | (athlete × stage × route) | route-by-route ascent data with all discipline-specific columns (`top`, `plus`, `time_ms`, `zone`, `points`, …) |

`gender` is translated from `0`/`1` to `"male"`/`"female"` inside the
views' SQL so downstream consumers don't need to know the encoding.

## 7. Run the export

`export_all(conn)` writes all seven default views to `data/exports/`
and returns a dict mapping view names to written paths. (`ascents` is
excluded — call `export_view(conn, "ascents")` explicitly if you want it.)

Each filename includes a UTC timestamp, so re-runs never overwrite a
previous export — you accumulate dated snapshots.

In [ ]:
from wcl_data.exporter import export_all

written = export_all(conn)
for name, path in written.items():
    size_kb = path.stat().st_size / 1024
    print(f"  {name:14s}  {path.name:50s}  ({size_kb:>9.1f} KB)")


### Exporting one view at a time

If you only want one CSV — say, after a `pull-new` you just want a
refreshed `results.csv` — use `export_view`:

In [ ]:
from wcl_data.exporter import export_view, VIEW_NAMES

print("Available views:", VIEW_NAMES)
path = export_view(conn, "results")
print(f"\nWrote: {path}")


## 8. Read a CSV back with pandas

The whole point of the CSV exports is that they're trivial to consume
elsewhere. Here we read the big `results` export back and explore it.

In [ ]:
# The most recent results export, by filename timestamp.
results_csv = max(Path("data/exports").glob("results_*.csv"))
print("reading:", results_csv.name)

results_df = pd.read_csv(results_csv)
print(f"shape: {results_df.shape}")
results_df.head()


In [ ]:
results_df.info()


In [ ]:
# Numeric summary — only `rank` is numeric in this view, but `describe`
# handles that gracefully.
results_df[["rank"]].describe()


### Example downstream slice

Now that you're holding a flat DataFrame, normal pandas idioms apply.
Here's the best result per athlete per discipline.

In [ ]:
best_per = (
    results_df
    .dropna(subset=["rank"])
    .groupby(["athlete_firstname", "athlete_lastname", "discipline"], as_index=False)["rank"]
    .min()
    .rename(columns={"rank": "best_rank"})
    .sort_values("best_rank")
)
best_per.head(10)


## 9. When to re-export

The exports are cheap to regenerate — typically under a second for the
non-`results` views and a few seconds for `results`. A reasonable
cadence:

- After every `pull-new` (so the exports include the new content).
- Before every analysis session, to be sure you're working on the
  latest snapshot.
- Never edit the CSVs in place — treat them as **derived artefacts**;
  the warehouse is the source of truth.

The whole `data/exports/` directory is in `.gitignore` because the
files are derived and timestamped — no reason to commit them.

## 10. Clean up

In [ ]:
conn.close()


## Where to go from here

You now have:

- a fully populated SQLite warehouse,
- seven denormalized CSV exports (plus opt-in `ascents`),
- a working understanding of the schema and the public Python API,
- enough joins under your belt to ask your own questions.

The warehouse is ready for downstream analysis — pick a question, write
the SQL (or the pandas), and have fun.

If you find yourself wanting a feature that isn't here yet, the source
under `src/wcl_data/` is small and consistently structured; chapter 02
covered every public class and function. The 69-test pytest suite under
`tests/` is also a good place to look for known-working call patterns
when you want to extend something.
